# Meilenstein 1: Daten-Integration und Qualitätssicherung
## Projekt: Predictive Analytics im Radsport – Grand Tour Stage Ranking

### Einleitung
Dieses Notebook bildet den ersten Schritt der Datenverarbeitungspipeline. Ziel ist es, die separat erhobenen Datensätze der **Renn-Ergebnisse** (Grand Tour Stages) und der **Fahrer-Biometrie** (Physiologische Daten & Spezialisierungen) zu einem konsistenten Master-Datensatz zu verschmelzen. (Quellen anfügen?!)

Ein präzises Ranking-Modell erfordert nicht nur die historischen Platzierungen, sondern auch das physische Profil der Athleten (Größe, Gewicht, Alter) sowie deren spezifische Stärken (Climbing, Time Trial, etc.). In diesem Prozess identifizieren wir zudem Datenlücken und bereinigen den Datensatz von Redundanzen, um eine saubere Grundlage für das spätere Feature Engineering und das Training des **XGBRanker-Modells** zu schaffen.

### Arbeitsschritte in diesem Notebook:
1. **Daten-Import:** Laden der Rohdaten im JSONL-Format.
2. **Daten-Merging:** Verknüpfung der Tabellen via `rider_url`.
3. **Data Cleaning:** Entfernung technisch irrelevanter Merkmale (z.B. Bild-URLs).
4. **Explorative Lücken-Analyse:** Identifikation und Visualisierung fehlender Werte (`NaN`).
5. **Checkpointing:** Speicherung des bereinigten Zustands als persistente Pickle-Datei.

---

### 1. Daten Import:
Laden der Rohdaten im JSONL Format

In [1]:
# Laden der nötigen Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:
# 0. Test Directory
#print("Aktuelles Verzeichnis:", os.getcwd())

# 1. DATEN-IMPORT
# Pfade definieren (Passe diese an, falls deine Dateien in Unterordnern liegen)
results_file = '../../data/raw/grand_tour_stage_results.jsonl'
riders_file = '../../data/raw/riders_full_biometrics.jsonl'

df_results = pd.read_json(results_file, lines=True)
df_riders = pd.read_json(riders_file, lines=True)

print(df_results.head(5))
print(100*"=")
print(df_riders['url_name'].head(10))


/var/folders/9x/rcpqk1vx40nfym40kyrwkkmc0000gn/T/ipykernel_16794/2452200725.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_results = pd.read_json(results_file, lines=True)


ValueError: Expected object or value

### 2. Vorbereitung Data Merging
- Explode (Zeilen-Vervielfältigung): Die Liste der Platzierungen in der Spalte stage_results wird "gesprengt", sodass aus einer Zeile pro Etappe ca. 150 bis 180 einzelne Zeilen werden – eine für jedes Fahrergebnis.
- Spalten Extraktion: Die in den Zeilen enthaltenen Dictionaries (mit Infos wie Platzierung, Zeit und Fahrer-URL) werden in flache, eigenständige Spalten umgewandelt, damit sie mathematisch auswertbar sind.
-Merge-Vorbereitung: Die extrahierte rider_url wird als eindeutiger Schlüssel (Key) genutzt, um im nächsten Schritt die körperlichen Merkmale aus der Biometrie-Tabelle passgenau an jedes einzelne Ergebnis anzuspielen.

In [18]:
df_results_long = df_results.explode('results').reset_index(drop=True) #Reset Index damit nicht alle Stage 1 Platzierungen gleichen Index haben
#print(df_results_long.head(5))
#Nun jeder Rank für jede Etappe eine Zeile

# Extraktion der Result Daten in einzelne Spalten
results_flat = pd.json_normalize(df_results_long['results'])
#results_flat.head(5)


# Zusammenführen der df_results_long (ohne results) mit results_flat
df_results_final = pd.concat([df_results_long.drop(columns=['results']),
                              results_flat], axis = 1)

df_results_final.head(10)

,race,year,url,metadata,rank,rider_url,time_gap
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",1,david-zabriskie,20:5120:51
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",2,lance-armstrong,0:020:02
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",3,alexandr-vinokourov,0:530:53
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",4,george-hincapie,0:570:57
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",5,laszlo-bodrogi,0:590:59
5,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",6,floyd-landis,1:021:02
6,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",7,fabian-cancellara,",,1:02"
7,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",8,jens-voigt,1:041:04
8,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",9,vladimir-karpets,1:051:05
9,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",10,igor-gonzalez-de-galdeano,1:061:06


### 3. Data Merging
- Tabellen-Merge (Left Join): Die Rennergebnisse werden über die rider_url mit der Biometrie-Tabelle verknüpft, sodass jedes Ergebnis um die Fahrer-Infos (Größe, Gewicht, etc.) ergänzt wird.


In [27]:
df_final = pd.merge(
    df_results_final,
    df_riders,
    left_on='rider_url',
    right_on='url_name',
    how='left'
)

print(df_final.head(5))

# Prüfung der fehlenden 88 Fahrerprofile
missing_rows = df_final['height'].isnull().sum()
print()

print(df_final.shape[0])
print(missing_rows)

# somit 32731 zeilen vorhanden
# 332 Zeilen nicht vorhanden -> Fahrerprofile fehlen

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

                                            metadata rank  \
0  {'departure': 'Fromentine', 'arrival': 'Noirmo...    1   
1  {'departure': 'Fromentine', 'arrival': 'Noirmo...    2   
2  {'departure': 'Fromentine', 'arrival': 'Noirmo...    3   
3  {'departure': 'Fromentine', 'arrival': 'Noirmo...    4   
4  {'departure': 'Fromentine', 'arrival': 'Noirmo...    5   

             rider_url    time_gap   birthdate  height  \
0      david-zabriskie  20:5120:51   1979-1-12    1.83   
1      lance-armstrong    0:020:02   1971-9-18  